In [ ]:
import os
from pathlib import Path
from PIL import Image, ImageEnhance
import cv2
import numpy as np

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models

import mlflow
import mlflow.pytorch

# --------------------------
# Préprocessing
# --------------------------
def preprocess_image(img_path, size=(224,224), apply_clahe=True, contrast_factor=1.2, brightness_factor=1.1):
    img = Image.open(img_path)
    if img.mode != "RGB":
        img = img.convert("RGB")
    img = img.resize(size)
    img_cv = np.array(img)
    if apply_clahe:
        lab = cv2.cvtColor(img_cv, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l = clahe.apply(l)
        lab = cv2.merge((l, a, b))
        img_cv = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    img = Image.fromarray(img_cv)
    img = ImageEnhance.Contrast(img).enhance(contrast_factor)
    img = ImageEnhance.Brightness(img).enhance(brightness_factor)
    return img

def preprocess_dataset(source_dir, target_dir, size=(224,224), apply_clahe=True):
    source_dir = Path(source_dir)
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    classes = [d.name for d in source_dir.iterdir() if d.is_dir()]
    print(f"Classes trouvées : {classes}")
    for cls in classes:
        cls_source = source_dir / cls
        cls_target = target_dir / cls
        cls_target.mkdir(exist_ok=True)
        images = [f for f in cls_source.glob("*.*") if not f.name.startswith(".")]
        for img_file in images:
            try:
                img = preprocess_image(img_file, size=size, apply_clahe=apply_clahe)
                img.save(cls_target / img_file.name)
            except Exception as e:
                print(f"Erreur traitement {img_file}: {e}")
    print(f"✅ Préprocessing terminé pour {source_dir}")

# --------------------------
# Chargement dataset
# --------------------------
def load_dataloaders(train_dir, val_dir, batch_size=32):
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    transform_val = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    train_dataset = datasets.ImageFolder(train_dir, transform=transform_train)
    val_dataset = datasets.ImageFolder(val_dir, transform=transform_val)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, len(train_dataset.classes)

# --------------------------
# CNN From Scratch
# --------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*28*28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# --------------------------
# ResNet50 Transfer Learning
# --------------------------
def get_resnet50(num_classes):
    model = models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

# --------------------------
# MobileNetV2 Transfer Learning
# --------------------------
def get_mobilenet_v2(num_classes):
    model = models.mobilenet_v2(pretrained=True)
    model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    return model

# --------------------------
# Entraînement commun
# --------------------------
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-4, device='cuda', experiment_name="experiment"):
    device = torch.device(device if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    mlflow.set_experiment(experiment_name)
    with mlflow.start_run():
        for epoch in range(epochs):
            model.train()
            running_loss, correct, total = 0.0, 0, 0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
            train_loss = running_loss / total
            train_acc = correct / total

            # Validation
            model.eval()
            val_loss, val_correct, val_total = 0.0, 0, 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item() * images.size(0)
                    _, predicted = torch.max(outputs.data, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()
            val_loss /= val_total
            val_acc = val_correct / val_total

            print(f"[{model.__class__.__name__}] Epoch {epoch+1}/{epochs} | "
                  f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("train_acc", train_acc, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("val_acc", val_acc, step=epoch)

        mlflow.pytorch.log_model(model, "model")
    return model

# --------------------------
# Runner
# --------------------------
if __name__ == "__main__":
    train_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/train"
    val_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/val"

    train_loader, val_loader, num_classes = load_dataloaders(train_dir, val_dir, batch_size=32)

    # CNN From Scratch
    cnn_model = SimpleCNN(num_classes)
    train_model(cnn_model, train_loader, val_loader, epochs=10, experiment_name="CNN_from_scratch_1")

    # ResNet50
    resnet_model = get_resnet50(num_classes)
    train_model(resnet_model, train_loader, val_loader, epochs=10, experiment_name="ResNet50_1")

    # MobileNetV2
    mobilenet_model = get_mobilenet_v2(num_classes)
    train_model(mobilenet_model, train_loader, val_loader, epochs=10, experiment_name="MobileNetV2_1")


In [ ]:
import mlflow
import mlflow.pytorch
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Experiments à récupérer
experiments = ["CNN_from_scratch_1", "ResNet50_1", "MobileNetV2_1"]

results = []

# Chargement des datasets (val_loader doit être le même que pour le training)
train_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/train"
val_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/val"
train_loader, val_loader, num_classes = load_dataloaders(train_dir, val_dir, batch_size=32)

for exp_name in experiments:
    # Récupérer le dernier run de l'expérience
    exp = mlflow.get_experiment_by_name(exp_name)
    if exp is None:
        print(f"Exp {exp_name} non trouvée")
        continue
    runs = mlflow.search_runs([exp.experiment_id], order_by=["start_time desc"], max_results=1)
    run_id = runs.iloc[0]["run_id"]

    # Charger le modèle depuis MLflow
    model = mlflow.pytorch.load_model(f"runs:/{run_id}/model")
    model = model.to(device)
    model.eval()

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    acc = (np.array(all_labels) == np.array(all_preds)).mean()
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")

    results.append({
        "model": exp_name,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

# Affichage final
import pandas as pd
df = pd.DataFrame(results)
print(df)
